<img src="images/logo.png" width=180, align="center"/>

Master's degree in Intelligent Systems

Subject: 11754 - Deep Learning

Year: 2025-2026

Professor: Miguel Ángel Calafat Torrens

# LESSON 4B — Training, Noise Schedules & Inference

This notebook builds on LESSON 4A. We cover the training algorithm, learning rate schedulers, different noise schedules, and inference with a pretrained model.

**Note: The code of this example is taken from [this repo](https://github.com/dome272/Diffusion-Models-pytorch) with license Apache 2.0 ([license](https://github.com/dome272/Diffusion-Models-pytorch/blob/main/LICENSE)), so this is also the license of this document.**

In [ ]:
# Environment detection: Colab vs local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running on {"Google Colab" if IN_COLAB else "local environment"}')

In [ ]:
# Setup: Drive mount (Colab) or local path
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    %cd '/content/gdrive/MyDrive/LABS2026/LAB04'

import logging
import os
import pathlib
import sys

PROJECT_DIR = str(pathlib.Path().resolve())
sys.path.append(PROJECT_DIR)

import helper_L4 as hp

import torch
from torch import nn, optim
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm

logging.basicConfig(format="%(asctime)s - %(levelname)s: %(message)s",
                    level=logging.INFO, datefmt="%I:%M:%S")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 1. The Training Objective

## Training Algorithm

One training step of a diffusion model:

1. Sample a clean image $x_0$ from the dataset
2. Sample a random timestep $t \sim \text{Uniform}(1, T)$
3. Sample random noise $\epsilon \sim \mathcal{N}(0, I)$
4. Compute the noisy image via reparameterization: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$
5. Predict the noise: $\epsilon_\theta(x_t, t)$
6. Compute loss: $L = \text{MSE}(\epsilon, \epsilon_\theta(x_t, t))$
7. Backpropagate and update weights

The key insight: the model **never sees clean images** during training. It only sees noisy versions at random timesteps and learns to predict what noise was added.

# 2. Dataset Setup

We use a subset of the [Landscape Pictures](https://www.kaggle.com/datasets/arnaud58/landscape-pictures) dataset (1000 images). Since the images are in a flat folder (no subdirectories), we use `hp.get_data_flat()` which wraps `hp.CustomDataset` — a dataset class that handles flat folder structures via recursive file search.

In [ ]:
# Dataset setup: extract from zip if needed
if IN_COLAB:
    dataset_zip = '/content/gdrive/MyDrive/datasets/landscape_pictures_1000.zip'
else:
    dataset_zip = os.path.join(PROJECT_DIR, '..', 'datasets', 'landscape_pictures_1000.zip')

DATASET_PATH = hp.extract_dataset(dataset_zip, remove_zip=IN_COLAB)
dataloader = hp.get_data_flat(DATASET_PATH, image_size=64, batch_size=4)

# 3. The Training Loop

Below is a clean training function adapted from the [dome272 repo](https://github.com/dome272/Diffusion-Models-pytorch). The original uses `argparse` and a global model variable — we've refactored it for notebook use with explicit parameters.

In [ ]:
def train(model, dataloader, diffusion, epochs=5, lr=3e-4, device=DEVICE,
          scheduler=None):
    """Train a diffusion model.

    Args:
        model: UNet model.
        dataloader: Training data.
        diffusion: Diffusion instance.
        epochs: Number of epochs.
        lr: Learning rate.
        device: Torch device.
        scheduler: Optional LR scheduler (created externally).

    Returns:
        list: Average loss per epoch.
    """
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    mse = nn.MSELoss()
    epoch_losses = []

    # If scheduler is provided as a class/constructor, instantiate it
    if scheduler is not None and callable(scheduler):
        scheduler = scheduler(optimizer)

    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        epoch_loss = 0.0
        n_batches = 0

        for images, _ in pbar:
            images = images.to(device)
            t = diffusion.sample_timesteps(images.shape[0]).to(device)
            x_t, noise = diffusion.noise_images(images, t)
            predicted_noise = model(x_t, t)
            loss = mse(noise, predicted_noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_postfix(MSE=loss.item())
            epoch_loss += loss.item()
            n_batches += 1

        if scheduler is not None:
            scheduler.step()

        avg_loss = epoch_loss / n_batches
        epoch_losses.append(avg_loss)
        logging.info(f"Epoch {epoch} avg loss: {avg_loss:.6f}")

    # Plot loss curve
    plt.figure(figsize=(8, 4))
    plt.plot(epoch_losses, marker='o')
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Average MSE Loss')
    plt.grid(True)
    plt.show()

    return epoch_losses

### Adapting Repository Code

Note how the original repo's `launch()` function uses `argparse.parse_args()` (which crashes in notebooks because Jupyter passes its own arguments) and a Windows-specific dataset path. We've replaced this with a clean function signature.

**In LAB 4, you will adapt the original repository code yourself.**

## Short Training Demo

Let's run 5 epochs to verify everything works. This is far too few for good results, but enough to see the loss trend downward.

In [ ]:
model = hp.UNet(device=DEVICE).to(DEVICE)
diffusion = hp.Diffusion(img_size=64, device=DEVICE)
losses = train(model, dataloader, diffusion, epochs=5)

# 4. Learning Rate Schedulers

Learning rate schedulers dynamically adjust the learning rate during training. A high learning rate helps early convergence; a lower rate later helps fine-tuning. See [PyTorch LR scheduler docs](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate).

## Categories of LR Schedulers

**Time-based** (predetermined schedule):
- `StepLR`: Reduce by factor `gamma` every `step_size` epochs
- `CosineAnnealingLR`: Smooth cosine decay (popular for diffusion)
- `ExponentialLR`: Exponential decay each epoch

**Performance-based** (adaptive):
- `ReduceLROnPlateau`: Reduce when a metric stops improving

**Cyclical** (oscillating):
- `OneCycleLR`: Single cosine cycle with warm-up
- `CosineAnnealingWarmRestarts`: Cosine with periodic restarts

## Implementation Pattern

<img src="images/ema.png" width=500, align="center"/>

**Key rule:** `scheduler.step()` must be called **after** `optimizer.step()`. This ordering was standardized in PyTorch 1.1.0.

`CosineAnnealingLR` is a popular choice for diffusion model training — it provides a smooth cosine decay of the learning rate. For warm restarts, use `CosineAnnealingWarmRestarts` instead.

# 5. Noise Schedules

The noise schedule $\{\beta_t\}_{t=1}^T$ controls how fast noise is added during the forward process. Different schedules produce different $\bar{\alpha}_t$ curves, which affect information decay rate and generation quality.

### Linear Schedule

The default from the original DDPM paper (Ho et al., 2020). Simple linear interpolation:

$$\beta_t = \beta_{\text{start}} + \frac{t}{T} (\beta_{\text{end}} - \beta_{\text{start}})$$

Known to destroy information too quickly at early timesteps.

### Cosine Schedule

Proposed by Nichol & Dhariwal (2021). Defines $\bar{\alpha}_t$ directly via a cosine function:

$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left(\frac{t/T + s}{1 + s} \cdot \frac{\pi}{2}\right)^2$$

Then $\beta_t$ is derived from consecutive $\bar{\alpha}_t$ values. This preserves more signal at early timesteps, producing smoother degradation. Implemented as `hp.CosineDiffusion`.

### Quadratic Schedule

More aggressive at later timesteps:

$$\beta_t = \beta_{\text{start}} + (\beta_{\text{end}} - \beta_{\text{start}}) \cdot t^2$$

### Sigmoid Schedule

S-shaped transition, concentrating changes in the middle of the process:

$$\beta_t = \beta_{\text{start}} + (\beta_{\text{end}} - \beta_{\text{start}}) \cdot \sigma(s \cdot (t - 0.5))$$

where $\sigma$ is the sigmoid function, normalized to span $[\beta_{\text{start}}, \beta_{\text{end}}]$.

In [ ]:
# Compare noise schedules: Linear vs Cosine
linear_diff = hp.Diffusion(noise_steps=1000, img_size=64, device='cpu')
cosine_diff = hp.CosineDiffusion(noise_steps=1000, img_size=64, device='cpu')

schedules = [
    ('Linear', linear_diff),
    ('Cosine', cosine_diff),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for label, d in schedules:
    beta = d.beta.cpu().numpy()
    alpha_hat = d.alpha_hat.cpu().numpy()
    snr = alpha_hat / (1 - alpha_hat + 1e-8)

    axes[0].plot(beta, label=label)
    axes[1].plot(alpha_hat, label=label)
    axes[2].plot(np.log(snr), label=label)

axes[0].set_title(r'$\beta_t$')
axes[0].set_xlabel('Timestep')
axes[0].legend()

axes[1].set_title(r'$\bar{\alpha}_t$')
axes[1].set_xlabel('Timestep')
axes[1].legend()

axes[2].set_title(r'$\log(\mathrm{SNR}_t)$')
axes[2].set_xlabel('Timestep')
axes[2].legend()

plt.tight_layout()
plt.show()

The **SNR (signal-to-noise ratio)** = $\bar{\alpha}_t / (1 - \bar{\alpha}_t)$ plot is the most informative. Notice how the cosine schedule preserves more signal at early timesteps (higher SNR for low $t$) and has a more uniform decay rate. The linear schedule drops too quickly at the beginning.

# 6. Inference

Let's load a pretrained model and generate images. The model was trained for 500 epochs on the landscapes dataset, from the [reference repo](https://github.com/dome272/Diffusion-Models-pytorch).

**Model setup:** Go to the reference repo's README, find the link to the pretrained models (Google Drive), and download `unconditional_ckpt.pt`. Save it to the `models/` folder so the path is `models/unconditional_ckpt.pt`.

In [ ]:
# Load pretrained model
model = hp.UNet(device=DEVICE).to(DEVICE)
ckpt = torch.load("./models/unconditional_ckpt.pt", map_location=DEVICE,
                   weights_only=True)
model.load_state_dict(ckpt)

diffusion = hp.Diffusion(img_size=64, device=DEVICE)

# Generate 8 images
x = diffusion.sample(model, 8)
hp.plot_images(x)

### Forward Diffusion Visualization (recap)

In [ ]:
hp.visualize_forward_diffusion(diffusion, dataloader, DEVICE)

### Reverse Diffusion Visualization

The reverse process step by step — from pure noise to a generated image:

In [ ]:
hp.visualize_reverse_diffusion(model, diffusion, DEVICE)

## Partial Denoising & Image-to-Image Generation

So far we've generated images starting from **pure noise** ($t = T-1$). But what if we start from a **real image** with some noise added?

The idea: take a clean image $x_0$, add noise up to an intermediate timestep $t$ using `noise_images()`, then run the reverse process from $t$ back to $0$. The result preserves the overall structure of the original image but introduces variation — the more noise added (higher $t$), the more "creative" the output.

This is the basis of **image-to-image generation (img2img)**: instead of creating from scratch, we *edit* an existing image by controlling how much noise to add. A low $t$ preserves most of the original; a high $t$ produces something almost entirely new.

In [ ]:
# Partial denoising demo: noise an image to different levels, then denoise
images, _ = next(iter(dataloader))
original = images[0:1].to(DEVICE)

# Use fractions of noise_steps for timestep values
noise_steps = diffusion.noise_steps
t_values = [noise_steps // 5, 2 * noise_steps // 5, 3 * noise_steps // 5, 4 * noise_steps // 5]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
orig_display = (original[0].clamp(-1, 1) + 1) / 2
axes[0].imshow(orig_display.cpu().permute(1, 2, 0).numpy())
axes[0].set_title('Original')
axes[0].axis('off')

was_training = model.training
model.eval()

for idx, t_start in enumerate(t_values):
    print(f"Denoising from t={t_start}...")
    t = torch.tensor([t_start], device=DEVICE)
    x_noisy, _ = diffusion.noise_images(original, t)
    # Denoise from t back to 0
    x = x_noisy.clone()
    with torch.no_grad():
        for i in reversed(range(1, t_start + 1)):
            ti = (torch.ones(1) * i).long().to(DEVICE)
            predicted_noise = model(x, ti)
            x = diffusion.denoise_step(x, i, predicted_noise)
    x = (x.clamp(-1, 1) + 1) / 2
    axes[idx + 1].imshow(x[0].cpu().permute(1, 2, 0).numpy())
    axes[idx + 1].set_title(f'From t={t_start}')
    axes[idx + 1].axis('off')

if was_training:
    model.train()

plt.suptitle('Partial denoising: more noise \u2192 more creative output')
plt.tight_layout()
plt.show()

# 7. Beyond DDPM

Several improvements have been proposed since the original DDPM:

- **EMA (Exponential Moving Average)**: Maintains a smoothed copy of model weights. The EMA model often produces better samples than the training model. See `hp.EMA`.

- **Conditional generation**: Class-conditioned UNet that generates images of a specific category. Uses class embeddings added to the time embedding. See `hp.UNet_conditional`.

- **DDIM** (Song et al., 2020): Deterministic sampling that produces images in far fewer steps (e.g., 50 instead of 1000). The key insight: the reverse process can be made deterministic by removing the stochastic noise term.

- **Classifier-free guidance**: Trades diversity for quality by amplifying the conditional signal during sampling. Used in DALL-E 2 and Stable Diffusion.

# 8. Summary

In this notebook we covered:

1. **Training**: Simple MSE loss between actual and predicted noise. Random timestep sampling makes it efficient.
2. **Learning rate schedulers**: CosineAnnealingLR is popular for diffusion training.
3. **Noise schedules**: Linear (original DDPM), cosine (improved), sigmoid, quadratic. The schedule controls how fast information is destroyed.
4. **Inference**: Load pretrained weights and sample via the iterative reverse process.
5. **Partial denoising**: Adding noise to a real image and denoising back produces controlled variations — the basis of img2img generation.

Now go to **LAB 4** to put these concepts into practice with hands-on exercises.